In [1]:
import csv
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = Path("sample_recording.csv")

with CSV_PATH.open("r", newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

if not rows:
    raise ValueError(f"No events found in {CSV_PATH.resolve()}")

print(f"Loaded {len(rows):,} events from {CSV_PATH.resolve()}")
print("Event types:", Counter(row.get("event", "") for row in rows))

FileNotFoundError: [Errno 2] No such file or directory: 'sample_recording.csv'

In [ ]:
def numeric_column(name):
    """Return a CSV column as a NumPy float array."""
    values = []
    for row in rows:
        try:
            value = float(row.get(name, ""))
        except (TypeError, ValueError):
            value = np.nan
        values.append(value)
    return np.asarray(values, dtype=float)


times = numeric_column("simulation_time")
packet_ids = numeric_column("packet_id")
sizes = numeric_column("size_bytes")
rtts = numeric_column("rtt_seconds")

events = np.asarray([row.get("event", "") for row in rows])
packet_types = np.asarray([row.get("packet_type", "") for row in rows])
reasons = np.asarray([row.get("reason", "") for row in rows])

valid_times = times[np.isfinite(times)]
if len(valid_times) == 0:
    raise ValueError("No valid simulation timestamps were found.")

start_time = float(np.min(valid_times))
end_time = float(np.max(valid_times))
duration = end_time - start_time

print(f"Simulation time range: {start_time:.3f}–{end_time:.3f} s")
print(f"Recorded duration: {duration:.3f} s")

In [ ]:
event_counts = Counter(events)

print("EVENT SUMMARY")
print("-" * 40)
for name, count in sorted(event_counts.items()):
    print(f"{name or '(empty event name)':24s} {count:8,d}")

sent_count = event_counts.get("packet_sent", 0)
delivered_count = event_counts.get("packet_delivered", 0)
drop_count = event_counts.get("packet_dropped", 0)
timeout_count = event_counts.get("timeout", 0)

print("\nBASIC METRICS")
print("-" * 40)
print(f"Requests/packets sent events: {sent_count}")
print(f"Successful response events:  {delivered_count}")
print(f"Packet drop events:          {drop_count}")
print(f"Request timeouts:            {timeout_count}")

if sent_count:
    print(f"Response events / sent events: {delivered_count / sent_count:.2%}")

# RTT samples are logged on successful responses.
valid_rtts = rtts[np.isfinite(rtts) & (rtts >= 0)]
if len(valid_rtts):
    print(f"\nRTT samples: {len(valid_rtts)}")
    print(f"Mean RTT:    {np.mean(valid_rtts):.6f} s")
    print(f"Median RTT:  {np.median(valid_rtts):.6f} s")
    print(f"95th percentile RTT: {np.percentile(valid_rtts, 95):.6f} s")
    print(f"Maximum RTT: {np.max(valid_rtts):.6f} s")

# Drop reasons
drop_reasons = Counter(
    reason for event, reason in zip(events, reasons)
    if event == "packet_dropped"
)
if drop_reasons:
    print("\nDROP REASONS")
    for reason, count in drop_reasons.most_common():
        print(f"{reason or '(unspecified)'}: {count}")

In [ ]:
plot_counts = Counter(events)
names = sorted(plot_counts)
counts = [plot_counts[name] for name in names]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(names, counts)
ax.set_title("Recording Event Counts")
ax.set_xlabel("Event type")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
valid = np.isfinite(times)
plot_times = times[valid]
plot_events = events[valid]

bin_count = min(50, max(1, len(np.unique(plot_times))))
if end_time > start_time:
    bin_edges = np.linspace(start_time, end_time, bin_count + 1)
else:
    bin_edges = np.array([start_time - 0.5, start_time + 0.5])

event_names = [
    name for name in ("packet_sent", "packet_delivered", "packet_dropped", "timeout")
    if name in set(plot_events)
]

fig, ax = plt.subplots(figsize=(11, 5))

for name in event_names:
    selected_times = plot_times[plot_events == name]
    counts, _ = np.histogram(selected_times, bins=bin_edges)
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    ax.plot(centers, counts, marker="o", markersize=3, label=name)

ax.set_title("Events Over Simulation Time")
ax.set_xlabel("Simulation time (s)")
ax.set_ylabel("Events per time bin")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:

valid_rtts = rtts[np.isfinite(rtts) & (rtts >= 0)]

if len(valid_rtts):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].hist(valid_rtts, bins="auto")
    axes[0].set_title("RTT Distribution")
    axes[0].set_xlabel("RTT (s)")
    axes[0].set_ylabel("Successful responses")
    axes[0].grid(alpha=0.3)

    rtt_mask = np.isfinite(times) & np.isfinite(rtts) & (rtts >= 0)
    order = np.argsort(times[rtt_mask])
    rtt_times = times[rtt_mask][order]
    sorted_rtts = rtts[rtt_mask][order]

    axes[1].scatter(rtt_times, sorted_rtts, s=12)
    axes[1].set_title("RTT Over Time")
    axes[1].set_xlabel("Simulation time (s)")
    axes[1].set_ylabel("RTT (s)")
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    plt.show()
else:
    print("No valid RTT samples. Check that successful responses are being recorded.")

In [ ]:
delivery_mask = (
    (events == "bytes_delivered")
    & np.isfinite(times)
    & np.isfinite(sizes)
    & (sizes >= 0)
)

delivery_times = times[delivery_mask]
delivery_bytes = sizes[delivery_mask]

if len(delivery_times):
    bin_width = 1.0  # seconds; adjust for your experiment
    edges = np.arange(start_time, end_time + bin_width, bin_width)

    if len(edges) < 2:
        edges = np.array([start_time, start_time + bin_width])

    bytes_per_bin, _ = np.histogram(
        delivery_times, bins=edges, weights=delivery_bytes
    )
    actual_widths = np.diff(edges)
    throughput_bps = bytes_per_bin * 8 / actual_widths
    centers = (edges[:-1] + edges[1:]) / 2

    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(centers, throughput_bps)
    ax.set_title(f"Delivered Throughput ({bin_width:g}s bins)")
    ax.set_xlabel("Simulation time (s)")
    ax.set_ylabel("Throughput (bits/s)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

    print(f"Total delivered bytes: {np.sum(delivery_bytes):,.0f}")
    if duration > 0:
        print(f"Average throughput: {np.sum(delivery_bytes) * 8 / duration:,.2f} bits/s")
else:
    print("No bytes_delivered events found.")